# Customer Support Multi-Agent Flow with CrewAI
## Sequential vs Hierarchical Execution

### Problem Statement
Customer support teams receive different kinds of requests—login issues, billing errors, refunds, technical failures, and general questions. A multi-agent system can divide this work among specialists.

This notebook uses four worker agents:

1. **Issue Identification Agent** — classifies intent, sentiment, and priority.
2. **Knowledge Base Agent** — finds the most relevant support information.
3. **Solution Generation Agent** — prepares a customer-friendly response.
4. **Escalation Agent** — decides whether a human should take over.

The notebook demonstrates **two different ways to control agent execution**:

- **Sequential process:** the developer fixes the order of tasks.
- **Hierarchical process:** a manager LLM coordinates/delegates work to specialist agents.

> Key idea: the **process** decides *how tasks are orchestrated*. The agent's role/goal/backstory helps determine *what that agent is good at*.


For running this notebook on your own machine, install the required packages:

```python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_openai python-dotenv
```

If you use a newer CrewAI version, the same concepts still apply, although some APIs may have evolved.


In [ ]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6


In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [ ]:
from crewai import Agent, Task, Crew, Process


In [ ]:
import os
import warnings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

warnings.filterwarnings("ignore")
load_dotenv()

# OPENAI_API_KEY should be available in your environment/.env file.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 1. Create the Worker Agents

Each worker has a **role**, **goal**, and **backstory**. These describe the specialist's capability.

For this teaching example, `allow_delegation=False` keeps the worker responsibilities clear.  
In hierarchical mode, the **manager** performs the delegation.


### Worker Agents
- `issue_classifier`
- `knowledge_agent`
- `solution_agent`
- `escalation_agent`


In [ ]:
issue_classifier = Agent(
    role="Issue Identification Specialist",
    goal="Correctly interpret and classify customer issues.",
    backstory="You are trained in identifying user intent, sentiment, urgency, and support category.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

knowledge_agent = Agent(
    role="Knowledge Base Specialist",
    goal="Find the best available support information for the customer's issue.",
    backstory="You are a support knowledge specialist who identifies the most relevant documented solution.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

solution_agent = Agent(
    role="Solution Generator",
    goal="Create a clear, helpful and customer-friendly support response.",
    backstory="You turn technical/support information into empathetic step-by-step guidance.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

escalation_agent = Agent(
    role="Escalation Coordinator",
    goal="Decide whether the issue can be resolved by AI or must be escalated to a human.",
    backstory="You assess urgency, uncertainty, missing information, and resolution quality.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

workers = [
    issue_classifier,
    knowledge_agent,
    solution_agent,
    escalation_agent
]


In [ ]:
# ------------------------------------------------------------
# TASK SET A: SEQUENTIAL
# Each task is explicitly assigned to one worker.
# Therefore, the developer controls both task order and ownership.
# ------------------------------------------------------------

identify_issue = Task(
    description="""Analyze the customer message: "{customer_query}".

Identify:
- Issue type (billing, technical, login, refund, general inquiry)
- Sentiment (angry, neutral, confused, urgent)
- Priority level (low, normal, high)

Return a concise structured result with:
type, urgency, sentiment, summary.""",
    agent=issue_classifier,
    expected_output="A structured classification containing type, urgency, sentiment, and summary."
)

fetch_knowledge = Task(
    description="""Using the customer message "{customer_query}" and the earlier classification,
find the closest matching support solution.

If no suitable solution is available, return exactly:
NO MATCH FOUND""",
    agent=knowledge_agent,
    expected_output="The best available support solution, or NO MATCH FOUND."
)

generate_solution_response = Task(
    description="""Using the customer message "{customer_query}", classification, and available
knowledge-base guidance, create a customer-facing response that:
- Uses plain language
- Gives clear next steps
- Includes an apology when appropriate
- Does not invent unsupported policy details""",
    agent=solution_agent,
    expected_output="A polished customer support response."
)

apply_escalation_logic = Task(
    description="""Review the earlier work and decide whether the issue is sufficiently resolved.

Consider:
- Urgency/priority
- Missing information
- Whether the knowledge step returned NO MATCH FOUND
- Whether the proposed solution is actionable
- Whether human intervention is safer or necessary

Return one of:
- resolved — with a short justification
- requires_escalation — with reason and escalation notes""",
    agent=escalation_agent,
    expected_output="Resolution status plus a short justification."
)

sequential_tasks = [
    identify_issue,
    fetch_knowledge,
    generate_solution_response,
    apply_escalation_logic
]


## 2. How Do We Decide the Execution Flow?

The important question is **not only "Which agent exists?"** but **"Who controls task assignment and order?"**

| Question | Sequential | Hierarchical |
|---|---|---|
| Is the task order known in advance? | Yes | Not necessarily |
| Who chooses the next worker? | Your Python/task list | Manager LLM |
| Is routing dynamic? | No / limited | Yes |
| Is there an extra manager reasoning step? | No | Yes |
| Cost/latency | Usually lower | Usually higher |
| Debugging/predictability | Easier | More adaptive, less deterministic |
| Best fit | Stable pipelines | Open-ended/dynamic coordination |

### For this customer-support example

A normal flow such as:

`Classify → Search Knowledge → Generate Response → Escalation Check`

is naturally **Sequential** because the dependency order is known.

Use **Hierarchical** when the incoming work is less predictable, for example:

`Customer Request → Manager decides specialist(s) → Delegates/reviews/reassigns → Final result`

Here the manager may decide that a billing specialist is needed first, or ask another specialist to re-check work before finishing.


In [ ]:
# ------------------------------------------------------------
# A small rule-based helper to RECOMMEND an execution process.
# This does not execute agents. It only makes the design choice explicit.
# ------------------------------------------------------------

def recommend_execution_mode(
    fixed_order=True,
    dynamic_specialist_selection=False,
    manager_review_needed=False,
    runtime_replanning_needed=False
):
    """Recommend 'sequential' or 'hierarchical' from workflow requirements."""

    if dynamic_specialist_selection or manager_review_needed or runtime_replanning_needed:
        return "hierarchical"

    if fixed_order:
        return "sequential"

    # When the workflow is not fixed, hierarchical is a safer default
    # for this teaching example.
    return "hierarchical"


# Example 1: Our current customer-support pipeline
mode_1 = recommend_execution_mode(
    fixed_order=True,
    dynamic_specialist_selection=False,
    manager_review_needed=False,
    runtime_replanning_needed=False
)

# Example 2: A manager must dynamically decide which specialist handles the work
mode_2 = recommend_execution_mode(
    fixed_order=False,
    dynamic_specialist_selection=True,
    manager_review_needed=True,
    runtime_replanning_needed=True
)

print("Fixed support pipeline  ->", mode_1)
print("Dynamic support routing ->", mode_2)


## 3. Sequential Execution

### Who decides the flow?
**We do, in Python.**

The sequence below is deterministic:

```text
Customer Query
     ↓
Issue Identification Agent
     ↓
Knowledge Base Agent
     ↓
Solution Generator
     ↓
Escalation Agent
     ↓
Final Result
```

CrewAI executes the tasks in the order provided to `tasks=[...]`.


In [ ]:
# ------------------------------------------------------------
# SEQUENTIAL CREW
# ------------------------------------------------------------

sequential_crew = Crew(
    agents=workers,
    tasks=sequential_tasks,
    process=Process.sequential,
    verbose=True
)

print("Sequential crew ready.")
print("Task order:")
for i, task in enumerate(sequential_tasks, start=1):
    print(f"{i}. {task.agent.role}")


In [ ]:
# Run the sequential crew

customer_query = (
    "I'm unable to log into my account even though I'm sure the password "
    "is correct. Can someone help?"
)

sequential_result = sequential_crew.kickoff(
    inputs={"customer_query": customer_query}
)

print("\n--- SEQUENTIAL FINAL RESPONSE ---\n")
print(sequential_result)


## 4. Hierarchical Execution

### Who decides the flow?
A **manager LLM** coordinates the workers.

Conceptually:

```text
                    ┌──────────────────────┐
Customer Query ───► │   Manager / Lead     │
                    └──────────┬───────────┘
                               │ decides/delegates
             ┌─────────────────┼─────────────────┐
             ↓                 ↓                 ↓
     Issue Specialist   Knowledge Specialist   Solution Specialist
             └─────────────────┼─────────────────┘
                               ↓
                     Escalation Specialist
                               ↓
                         Manager Review
                               ↓
                           Final Result
```

For a hierarchical process, we give CrewAI the worker agents plus a `manager_llm`.  
The hierarchical task definitions below intentionally **do not hard-code an agent** so the manager can select the appropriate worker.


In [ ]:
# ------------------------------------------------------------
# TASK SET B: HIERARCHICAL
#
# Important difference:
# No worker is assigned with agent=...
# The manager uses worker roles/goals/backstories to decide delegation.
# ------------------------------------------------------------

h_identify_issue = Task(
    description="""Analyze the customer message: "{customer_query}".
Determine issue type, sentiment, urgency, priority, and a short summary.""",
    expected_output="A structured issue classification."
)

h_fetch_knowledge = Task(
    description="""Find the most relevant support guidance for "{customer_query}".
Use the available issue understanding from the crew.
If no suitable solution is available, return NO MATCH FOUND.""",
    expected_output="Relevant support guidance or NO MATCH FOUND."
)

h_generate_solution = Task(
    description="""Prepare a clear and empathetic customer-facing solution for "{customer_query}".
Use useful findings produced by the crew and provide actionable next steps.""",
    expected_output="A polished customer support response."
)

h_escalation_check = Task(
    description="""Assess whether the customer's issue has been safely and sufficiently resolved.
If not, identify why human escalation is required.""",
    expected_output="resolved or requires_escalation, with justification."
)

hierarchical_tasks = [
    h_identify_issue,
    h_fetch_knowledge,
    h_generate_solution,
    h_escalation_check
]


In [ ]:
# ------------------------------------------------------------
# HIERARCHICAL CREW
#
# manager_llm creates the management layer.
# The manager plans/delegates/reviews the work performed by workers.
# ------------------------------------------------------------

hierarchical_crew = Crew(
    agents=workers,
    tasks=hierarchical_tasks,
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True
)

print("Hierarchical crew ready.")
print("The manager LLM will coordinate/delegate work to the worker agents.")


In [ ]:
# Run the hierarchical crew

hierarchical_result = hierarchical_crew.kickoff(
    inputs={"customer_query": customer_query}
)

print("\n--- HIERARCHICAL FINAL RESPONSE ---\n")
print(hierarchical_result)


## 5. One Switch to Run Either Mode

This is useful when teaching or testing the same use case.

Change only:

```python
EXECUTION_MODE = "sequential"
```

to:

```python
EXECUTION_MODE = "hierarchical"
```

The important architectural difference remains:

- **Sequential:** task order and agent ownership are encoded by the developer.
- **Hierarchical:** manager reasoning controls delegation to workers.


In [ ]:
# ------------------------------------------------------------
# CHOOSE THE EXECUTION MODE
# ------------------------------------------------------------

EXECUTION_MODE = "sequential"   # "sequential" or "hierarchical"


def select_crew(execution_mode):
    execution_mode = execution_mode.strip().lower()

    if execution_mode == "sequential":
        print(
            "Selected SEQUENTIAL: fixed pipeline; "
            "the task list decides the execution order."
        )
        return sequential_crew

    if execution_mode == "hierarchical":
        print(
            "Selected HIERARCHICAL: manager LLM coordinates "
            "and delegates to specialists."
        )
        return hierarchical_crew

    raise ValueError(
        "EXECUTION_MODE must be either 'sequential' or 'hierarchical'."
    )


selected_crew = select_crew(EXECUTION_MODE)

selected_result = selected_crew.kickoff(
    inputs={"customer_query": customer_query}
)

print("\n--- SELECTED MODE FINAL RESPONSE ---\n")
print(selected_result)


## 6. Practical Decision Rule

Use **Sequential** when:
- The business process already has a clear order.
- Every step has a known owner.
- You want predictable execution and easier debugging.
- Lower coordination cost/latency matters.

Use **Hierarchical** when:
- The correct specialist cannot be known until runtime.
- A lead/manager must delegate work dynamically.
- Tasks may need re-assignment, review, or replanning.
- The problem is open-ended enough to justify manager overhead.

### Recommended mode for this notebook
For the four-step customer-support pipeline as currently designed, **Sequential is the better default** because the dependencies are already known.

Hierarchical is included to show how the same group of specialists can be managed dynamically when the support workflow becomes more open-ended.


## 7. Try It Yourself

Test queries:

```text
This is urgent. My business is down because your system will not sync orders. Fix this ASAP.

I was charged twice for my subscription this month. Please refund the extra amount.

The app crashes every time I open the dashboard. This started after the latest update.

Does your software support exporting data to Excel? I cannot find the option.
```

Try each query once with `EXECUTION_MODE = "sequential"` and once with `EXECUTION_MODE = "hierarchical"` and compare:
- which agent is used,
- execution trace,
- manager/delegation behavior,
- final answer,
- latency/token usage.
